# Demo: GNN-BERT Music Context Understanding

One end-to-end inference example (Task 3 fusion model): given a track's
structure graph + a text caption, predict tags and valence/arousal.

Run `python src/make_toy_dataset.py`, `python src/audio_features.py` (real data)
or the toy generator, `python src/graph_builder.py`, and `python src/train.py --task 3`
before running this notebook.

In [ ]:
import sys, json
sys.path.insert(0, '../src')
import torch
import numpy as np
import yaml
from transformers import AutoTokenizer
from fusion_model import GNNBertFusionModel
from datasets import load_tag_vocab, load_manifest

with open('../config.yaml') as f:
    cfg = yaml.safe_load(f)
tags = load_tag_vocab(toy_dir='../data/processed/toy')
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(cfg['model']['bert_name'])
test_items = load_manifest('test', splits_dir='../data/splits')
example = test_items[0]
graph = np.load(f"../data/processed/graphs/{example['track_id']}.npz")
in_dim = graph['seg_x'].shape[1]

model = GNNBertFusionModel(
    bert_name=cfg['model']['bert_name'], gnn_in_dim=in_dim,
    gnn_hidden=cfg['model']['gnn_hidden'], gnn_layers=cfg['model']['gnn_layers'],
    num_tags=len(tags), fusion_type=cfg['model']['fusion_type'],
).to(device)
model.load_state_dict(torch.load('../results/task3_fusion.pt', map_location=device))
model.eval()

In [ ]:
enc = tokenizer(example['caption'], truncation=True, padding='max_length', max_length=cfg['data']['max_text_len'], return_tensors='pt')
x = torch.tensor(graph['seg_x'], dtype=torch.float32)
edge_index = torch.tensor(graph['seg_edge_index'], dtype=torch.long)
batch_index = torch.zeros(x.shape[0], dtype=torch.long)

with torch.no_grad():
    tag_logits, va_pred = model(
        enc['input_ids'].to(device), enc['attention_mask'].to(device),
        x.to(device), edge_index.to(device), batch_index.to(device),
    )
    probs = torch.sigmoid(tag_logits).cpu().numpy()[0]

print('Caption:', example['caption'])
print('True tags:', [t for t, v in zip(tags, example['tags']) if v])
print('Predicted tags (p>0.5):', [t for t, p in zip(tags, probs) if p > 0.5])
print('Predicted valence/arousal:', va_pred.cpu().numpy()[0])
print('True valence/arousal:', example['valence'], example['arousal'])